# Task: Peak Finding Exercise

Try to complete as much of this notebook as you can in the time available. It is OK if you don't complete everything.

As you complete each task, paste the result and code into this form:

https://form.menti.com/s/alvjkrkfkiev

At the end of the allotted time, submit the form so we can see how far you got through the tasks. This is for the purposes of the educational study - remember that you are completely anonymous.

If you finish early, use the extra time to make sure you understand all of the code you have produced.

---




When dealing with a series of values, we might commonly want to know what the maximum value is. For example, if we were looking at the temperature of a sample in a chemistry lab, we might want to know what the maximum temperature was during the course of an experiment. We might have some data which looks something like this:

<center><img src="../assets/peak_full.png" style="height:300px" /></center>

In this case, the data has been taken every 1 hour, however, its clear that this is not often enough to capture all the variation of the actual temperature. For example, we can zoom in on the time period between 3 and 7 hours and plot the actual temperature alongside the data points recorded:

<center><img src="../assets/peak_closeup.png" style="height:300px" /></center>

As we can see, the data points do not capture the full variation of the temperature over time. In particular, the data points do not capture the peak temperature or the timing of this peak. In some experiments ot simulations, capturing these features of a data set can be very useful.

In this exercise, you will build a code which can interpolate between points of a data set to find the peak value and the time at which this peak occurs. You will start by considering only three points at a time and then extend your approach to the entire dataset. Each three points will be used to fit a quadratic curve, which will then be used to estimate the peak within that segment.


## Breaking the task down to steps

When planning to tackle a task—large or small—it is a good idea to outline the structure of the code (there is no coding involved yet). In this task, we build the code from the bottom up. 

Before considering the whole dataset, we'll concentrate on solving the task for three points. Once we can find the peak for three points, we will apply this function to the whole dataset by looping over all points. 

This notebook divides the development process into three tasks. Each task description includes the needed information regarding the math and coding approaches. This is an example approach to tackling this problem.

**Task 1** - Write a function that takes the coordinates of three points and returns quadratic coefficients. These coefficients will define the quadratic curve that passes through the three points.

**Task 2** - Use the function from Task 1 and find a peak (or maximum) for three points.

**Task 3** - Apply the function from Task 2 to the whole dataset (all points of your measurement) to capture the overall peak of the dataset.

## Task 1 - The Quadratic Approximation

We can approximate the shape of a curve around a point by defining a quadratic function which passes through the points an the points on either side of it. For the peak we're discussing, this function may look like this:c

<center><img src="../assets/peak_fit.png" style="height:300px" /></center>

As we can see, the fit still isn't perfect, but we are getting an improved estimate of the peak value and the time at which this peak occurs. The quadratic fit has the equation:

$$ y = ax^{2} + bx + c $$

where $a$, $b$ and $c$ are the quadratic, linear, and constant coefficients of the function. We can find these coefficients by solving the following equations:

$$ a = \frac{x_{1}(y_{3} - y_{2}) + x_{2}(y_{1} - y_{3}) + x_{3}(y_{2} - y_{1})}{(x_{1} - x_{2})(x_{1} - x_{3})(x_{2} - x_{3})} $$

$$ b = \frac{y_{2} - y_{1}}{x_{2} - x_{1}} - a (x_{1} + x_{2})$$

$$ c = y_{1} - ax_{1} ^{2} - bx_{1} $$

where $(x_{1}, y_{1})$, $(x_{2}, y_{2})$ and $(x_{3}, y_{3})$ are the Cartesian coordinates of the point to the left, the central point, and the point to the right of the region we're approximating.

### Coding

Using the equations for $a$, $b$, and $c$ above, write a function in the code cell below named ```get_quadratic_coefficients``` which receives 6 arguments. These arguments should be, in order, $x_{1}$, $y_{1}$, $x_{2}$, $y_{2}$, $x_{3}$, and $y_{3}$. The function should return a list with three entries, containing the values of $a$, $b$, and $c$. There are a few calls to the function in the cell below, which you can use to test your function.

In [ ]:
# Write your function here
def get_quadratic_coefficients(x1, y1, x2, y2, x3, y3):
    # ...


### Test get_quadratic_coefficients

In the code cell below write test for get_quadratic_coefficients. 
<pre>
- get_quadratic_coefficients(-1, 0, 0, 0, 1, 0)  should return [0.0, 0.0, 0.0]
- get_quadratic_coefficients(4.2, -10.1, 4.4, -9.5, 4.6, -8.9)) should return [0.0, 3.0, -22.7]
- get_quadratic_coefficients(-10, 200, 0, 0, 10, 200) should return [2.0, 0.0, 0.0]
- get_quadratic_coefficients(4, 10, 5, 20, 6, 18) should return [-6.0, 64.0, -150.0]
</pre>

You will notice that floating point arithmetic can sometimes lead to small rounding errors in the results. You cannot use exact equality checks for floating point numbers. Instead, you can use function ```pytest.approx``` to compare floating point numbers (or lists of floating point numbers). 

In [ ]:
import ipytest
import pytest
ipytest.autoconfig()

In [ ]:
%%ipytest

# write your test here


When the tests show that your function is working correctly, run this cell to get your answer to part 1:

In [ ]:
get_quadratic_coefficients(2.5, 10.0, 5.0, 20.0, 7.5, 8.0)

## Task 2 - Finding the Peak

Once we have the coefficients of the quadratic function, we can find the peak by finding the $x$ value at which the function reaches its maximum. One way to do this is by finding the value of $x$ at which the gradient of the function is equal to zero. The gradient of the function is given by:

$$ g = 2ax + b $$

and is equal to zero when:

$$ 0 = 2a x_{flat} + b $$

$$ x_{flat} = -\frac{b}{2a} $$

Considering general properties of quadratic function, if the value of $a$ is positive, then this value of $x_{flat}$ corresponds to a minimum, and if the value of $a$ is negative, then this value of $x_{flat}$ corresponds to a maximum. If the value of $a$ is zero, then the quadratic coefficient is zero and the function has no peak. So, the peak of the quadratic function (if it exists) occurs at:

$$ x_{peak} = x_{flat} = -\frac{b}{2a} \quad \text{if } a < 0 $$

### Coding

Write a function named ```get_peak_in_region``` which finds the peak of the function approximated by three data-points. This function should receive 6 arguments. These arguments should be, in order, $x_{1}$, $y_{1}$, $x_{2}$, $y_{2}$, $x_{3}$, and $y_{3}$, where $(x_{1}, y_{1})$, $(x_{2}, y_{2})$ and $(x_{3}, y_{3})$ are the Cartesian coordinates of the point on the left boundary left, the central point, and the point on the right boundary of the region we're approximating. The function will return coordinates $(x_{peak}, y_{peak})$ for peak, if it exists.

This function should use your function ```get_quadratic_coefficients``` to find the coefficients of the quadratic function which approximates the data. 

If $a$ is negative, it should then use these coefficients to find the $x$ value at which the gradient of the function is equal to zero.  If this $x_{peak}$ is between $x_{1}$ and $x_{3}$, then the quadratic function contains a peak. Otherwise, the function should return ```None```. You can write a separate function for peak finding ```get_flat_x``` to find the $x_{peak}$.

If we have the $x_{peak}$, we need to calculate the corresponding $y_{peak}$ by evaluating the quadratic function at $x_{peak}$. For this, it is useful to have a separate function that evaluates a quadratic function given its coefficients and an $x$ value. Let's call this function ```evaluate_quadratic_function```.

Overall, the function should return the coordinates of the peak as a list ```[x_peak, y_peak]``` if a peak exists within the region, or ```None``` otherwise.


In [ ]:
# Write your functions here
# get_flat_x finds the x-coordinate of the peak (or flat point) of a quadratic function given its coefficients a and b.
# evaluate_quadratic_function evaluates a quadratic function at a given x-coordinate using its coefficients a, b, c, and coordinate x.
# get_peak_in_region finds the peak of the quadratic function within the specified region defined by three data points.

def get_flat_x(a, b):
    # ...

def evaluate_quadratic_function(a, b, c, x):
    # ...

def get_peak_in_region(x1, y1, x2, y2, x3, y3):
    # ...


### Test get_peak_in_region
<pre>
- get_peak_in_region(0.8, 0, 1, 1, 1.2, 0) should return [1.0, 1.0]
- get_peak_in_region(3, -9, 4, -9, 5, -12) should return [3.5, -8.625]
- get_peak_in_region(5, 1.2, 5.1, 1.3, 5.2, 1.4) should return None
- get_peak_in_region(1, 1, 2, 0, 3, -2) should return None
- get_peak_in_region(-10, 0, -5, 10, 0, 15) should return None
- get_peak_in_region(0, 2, 1, 1, 2, 2) should return None
</pre>

In [ ]:
%%ipytest
# Test get_peak_in_region function here

   

When the tests show that your functions are working correctly, run this cell to get your answer for part 2:

In [16]:
get_peak_in_region(2.5, 10.0, 5.0, 20.0, 7.5, 8.0)

[4.886363636363636, 20.022727272727273]

## Task 3 - Examining a Whole Dataset

Now that we have a function which can determine if there is a peak in a region of three data points and where that peak is, we can use this function to examine a whole dataset and find which is the highest peak.

To do this, we will consider each region of three data points in turn, and use our function ```get_peak_in_region``` to determine if there is a value in each region. If there is, we will evaluate if it higher than the highest peak we have found so far. If it is, we will store the coordinates of this peak.

For example, if we have data points $(x_{1}, y_{1}), (x_{2}, y_{2}), ... , (x_{10}, y_{10})$ We will first examine the region between $x_{1}$ and $x_{3}$, then the region between $x_{2}$ and $x_{4}$, then the region between $x_{3}$ and $x_{5}$, and so on.

### Coding

Write a function named ```get_peak_dataset```. This should receive two arguments. The first is a list containing the x values of the data set, and the second is a list containing the y values of the data set. The function should return a list containing the Cartesian coordinates of the highest peak in the data set. If the dataset does not contain any peaks, the function should return ```None```.

You may assume that the two lists provided as arguments will be the same length, will contain only numerical values, and that the data set will contain at least three points.

Write your function in the cell below. There are some calls to the function in the cell which you can use to test your function. In this task, you do not need to write any other auxiliary functions.

In [ ]:
# Write your function here


### Test get_peak_dataset

<pre>
x = [0, 1, 2]
y = [0, 1, 0]
get_peak_dataset(x, y) should return the value [1.0, 1.0]

x = [0, 1, 2, 3, 4]
y = [0, 1, 0, 2, 0]
get_peak_dataset(x, y) should return the value [3.0, 2.0]

x = [0, 1, 2, 3, 4]
y = [1, 3.5, 1, 3, 1]
get_peak_dataset(x, y) should return the value [1.0, 3.5]

x = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
y = [0, 1, 2, 5, 10, 20, 18, 14, 12, 11, 10, 10, 11, 13, 15, 17, 19, 20.5, 19, 17, 14]
get_peak_dataset(x, y) should return the value [5.33.., 20.66...]

x = [0, 1, 2, 3, 4]
y = [0, -1, -2, -1, 0]
get_peak_dataset(x, y) should return None as there is no peak
</pre>

In [ ]:
%%ipytest

# Test get_peak_dataset function here, do not forget to use pytest.approx
# you will notice that even with pytest.approx, one test fails. 
# In this case, use additional arguments with pytest.approx([value1, value2],rel=1e-2) to specify higher relative tolerance.



When you are confident that the function is working correctly, run this cell to obtain your solution for part 3:

In [ ]:
get_peak_dataset([1,2,3,4,5,6,7],[82.8,83.3,80.0,80.4,67.5,75.7,71.1])

## Task 4. Visualisation

To make our analysis more practically useful, we can add a visualisation of the dataset, along with a marker to show the position of the calculated maximal peak and a label giving its coordinates.

We will use the `matplotlib` package for data graphics (https://matplotlib.org), which can accept coordinate lists like the ones we are using.

Import the `matplotlib.pyplot` module as shown below:

In [ ]:
# plt is a standard abbreviation for this module
import matplotlib.pyplot as plt  

Investigate how to use the [`scatter()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html), [`vlines()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.vlines.html) and [`text()`](https://matplotlib.org/stable/api/text_api.html#module-matplotlib.text) functions from this module. Can you write a function `plot_peak()` that accepts x and y coordinates as two lists, runs `get_peak_dataset()` and plots the results?

You can make the plot as simple or as fancy as you like.

In [ ]:
x = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
y = [0, 1, 2, 5, 10, 20, 18, 14, 12, 11, 10, 10, 11, 13, 15, 17, 19, 20.5, 19, 17, 14]


The file `data/peak_data.csv` contains a series of x and y coordinates.
Read in the data and construct two lists for the x and y values.



Use your plot_peak() function to visualise the data.

What are the coordinates of the peak for this dataset? This is your answer for part 4.